# FuturePath AI - Model Training Pipeline
This notebook loads the student career profile dataset, performs data cleaning and preprocessing, and trains two key models:
1. **Random Forest Classifier**: For multi-class job role prediction.
2. **Nearest Neighbors (KNN)**: For Case-Based Reasoning (CBR) student similarity queries.

Finally, it saves the models and encoders to `career_model_data.pkl` for hot-loading in our Streamlit web application.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle
import os

### 1. Load Dataset

In [ ]:
DATA_PATH = "PS2_Dataset.csv"
df = pd.read_csv(DATA_PATH)

# Clean column names (strip trailing spaces)
df.columns = df.columns.str.strip()
print("Dataset Shape:", df.shape)
df.head()

### 2. Define Features & Target Split

In [ ]:
X = df.drop(columns=['Suggested Job Role'])
y = df['Suggested Job Role']

### 3. Preprocessing & Encoding

In [ ]:
# Define column classifications
binary_cols = ['self-learning capability?', 'Extra-courses did', 'Taken inputs from seniors or elders', 'worked in teams ever?', 'Introvert']
rating_cols = ['reading and writing skills', 'memory capability score']
nominal_cols = [
    'certifications', 'workshops', 'Interested subjects', 'interested career area',
    'Type of company want to settle in?', 'Interested Type of Books',
    'Management or Technical', 'hard/smart worker'
]
numeric_cols = ['Logical quotient rating', 'hackathons', 'coding skills rating', 'public speaking points']

# Clean binary columns
for col in binary_cols:
    X[col] = X[col].str.lower().str.strip().map({'yes': 1, 'no': 0}).fillna(0).astype(int)
    
# Clean rating columns
for col in rating_cols:
    X[col] = X[col].str.lower().str.strip().map({'poor': 0, 'medium': 1, 'excellent': 2}).fillna(1).astype(int)
    
# Ordinal Encoder for nominal categorical columns
nominal_categories = {}
for col in nominal_cols:
    X[col] = X[col].str.strip()
    nominal_categories[col] = sorted(X[col].unique())
    
encoder = OrdinalEncoder(
    categories=[nominal_categories[col] for col in nominal_cols],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

X_encoded_nominal = encoder.fit_transform(X[nominal_cols])
X_processed = X.copy()
for i, col in enumerate(nominal_cols):
    X_processed[col] = X_encoded_nominal[:, i]
    
# Ensure numeric types
for col in numeric_cols:
    X_processed[col] = pd.to_numeric(X_processed[col]).fillna(0).astype(float)
    
# Target encoding
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Preprocessing complete. Shape:", X_processed.shape)

### 4. Train-Test Split and Model Performance Evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

eval_rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
eval_rf.fit(X_train, y_train)
y_pred = eval_rf.predict(X_test)

print(f"Generalization Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

### 5. Final Model Training (Full Dataset)

In [ ]:
print("Training Random Forest on full dataset...")
rf_model = RandomForestClassifier(n_estimators=150, max_depth=20, random_state=42)
rf_model.fit(X_processed, y_encoded)

print("Training Nearest Neighbors (KNN) CBR Matcher...")
knn_model = NearestNeighbors(n_neighbors=5, metric='cosine')
knn_model.fit(X_processed)

### 6. Export Models & Mappings

In [ ]:
categorical_mappings = {
    'binary': ['No', 'Yes'],
    'ratings': ['Poor', 'Medium', 'Excellent'],
    'nominal': nominal_categories
}

model_data = {
    'rf_model': rf_model,
    'knn_model': knn_model,
    'encoder': encoder,
    'label_encoder': label_encoder,
    'categorical_mappings': categorical_mappings,
    'feature_names': list(X.columns),
    'binary_cols': binary_cols,
    'rating_cols': rating_cols,
    'nominal_cols': nominal_cols,
    'numeric_cols': numeric_cols,
    'X_processed': X_processed,
    'y_encoded': y_encoded,
    'df_raw': df
}

OUTPUT_PATH = "career_model_data.pkl"
with open(OUTPUT_PATH, 'wb') as f:
    pickle.dump(model_data, f)

print(f"Model files exported successfully to {OUTPUT_PATH}!")